In [1]:
import pandas as pd
import numpy as np

In [7]:
df=pd.read_csv("C:\\Users\\Kirti\\OneDrive\\Desktop\\Fake Job Posting Detection & Analytics Dashboard\\data\\fake_real_job_postings.csv")
df.head()

,job_id,job_title,job_description,requirements,benefits,company_name,company_profile,industry,employment_type,location,...,application_deadline,contact_email,company_website,has_logo,num_open_positions,job_function,telecommuting,fraud_reason,text_length,is_fake
0,1,Software Engineer,We are looking for responsibilities fast-paced...,Candidates should have dynamic team skills fas...,We offer required skills fast-paced skills req...,Company_543,Our company growth fast-paced responsibilities...,Marketing,Contract,"Toronto, Canada",...,2024-09-16,hr312@company.com,https://www.company.com,0,3,Management,0,NaN,89,0
1,2,Content Writer,We are looking for required support experience...,Candidates should have required team fast-pace...,We offer fast-paced dynamic dynamic strategy g...,Company_192,Our company fast-paced opportunity innovation ...,Finance,Full-time,"Toronto, Canada",...,2024-10-18,hr127@company.com,https://www.company.com,0,10,Development,1,NaN,89,0
2,3,Customer Support Specialist,We are looking for dynamic required fast-paced...,Candidates should have preferred knowledge opp...,We offer skills experience required growth res...,NaN,We are global innovation growth skills knowled...,Healthcare,Internship,Remote,...,2024-01-13,job92@gmail.com,NaN,0,6,Support,0,Suspicious email,69,1
3,4,Data Analyst,We are looking for collaboration skills suppor...,Candidates should have innovation team require...,We offer strategy strategy dynamic support opp...,Company_95,Our company fast-paced support team strategy i...,Healthcare,Part-time,"Berlin, Germany",...,2024-02-09,hr366@company.com,https://www.company.com,1,4,Management,1,NaN,89,0
4,5,Graphic Designer,We are looking for team growth growth fast-pac...,Candidates should have experience preferred kn...,We offer opportunity skills responsibilities c...,NaN,We are global experience skills preferred fast...,Retail,Part-time,"London, UK",...,2024-08-26,job359@gmail.com,NaN,0,5,Management,0,No salary info,69,1


In [8]:
df.shape


(3000, 25)

In [9]:
df.columns

Index(['job_id', 'job_title', 'job_description', 'requirements', 'benefits',
       'company_name', 'company_profile', 'industry', 'employment_type',
       'location', 'salary_range', 'required_experience_years',
       'education_level', 'department', 'posting_date', 'application_deadline',
       'contact_email', 'company_website', 'has_logo', 'num_open_positions',
       'job_function', 'telecommuting', 'fraud_reason', 'text_length',
       'is_fake'],
      dtype='object')

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   job_id                     3000 non-null   int64 
 1   job_title                  3000 non-null   object
 2   job_description            3000 non-null   object
 3   requirements               3000 non-null   object
 4   benefits                   3000 non-null   object
 5   company_name               1528 non-null   object
 6   company_profile            3000 non-null   object
 7   industry                   3000 non-null   object
 8   employment_type            3000 non-null   object
 9   location                   3000 non-null   object
 10  salary_range               3000 non-null   object
 11  required_experience_years  3000 non-null   int64 
 12  education_level            3000 non-null   object
 13  department                 3000 non-null   object
 14  posting_

In [12]:
df.isnull().sum()

job_id                          0
job_title                       0
job_description                 0
requirements                    0
benefits                        0
company_name                 1472
company_profile                 0
industry                        0
employment_type                 0
location                        0
salary_range                    0
required_experience_years       0
education_level                 0
department                      0
posting_date                    0
application_deadline            0
contact_email                   0
company_website              1472
has_logo                        0
num_open_positions              0
job_function                    0
telecommuting                   0
fraud_reason                 1528
text_length                     0
is_fake                         0
dtype: int64

Cleaning Data


In [15]:
df = df.drop(['job_id'], axis=1)



Handling Missing Values

In [17]:
df['company_website']=df['company_website'].fillna('No Website')

In [21]:
df['company_name']=df['company_name'].fillna('Unknown')

In [22]:
df['fraud_reason']=df['fraud_reason'].fillna('Not Specified')

In [23]:
df.isnull().sum()

job_title                    0
job_description              0
requirements                 0
benefits                     0
company_name                 0
company_profile              0
industry                     0
employment_type              0
location                     0
salary_range                 0
required_experience_years    0
education_level              0
department                   0
posting_date                 0
application_deadline         0
contact_email                0
company_website              0
has_logo                     0
num_open_positions           0
job_function                 0
telecommuting                0
fraud_reason                 0
text_length                  0
is_fake                      0
dtype: int64

Convert Text to Lowercase

In [ ]:
text_cols=['job_title','job_description','requirements','benefits','company','company_profile']

for col in text_cols:
    df[col]=df[col].str.lower()

Converting date columns

In [24]:
df['posting_date']=pd.to_datetime(df['posting_date'])
df['application_deadline']=pd.to_datetime(df['application_deadline'])


Creating new columns

In [27]:
df['desc_length']=df['job_description'].apply(len)

In [28]:
df['has_website']=df['company_website'].apply(lambda x:0 if x=='No Website' else 1)

In [31]:
fake_keywords = ['easy money', 'quick cash', 'no experience', 'work from home','earn fast', 'limited time', 'guaranteed income', 'make money online']

df['keyword_flag']=df['job_description'].apply(lambda x: sum(word in x for word in fake_keywords))

Posting delay


In [40]:
df['posting_gap_days']=(df['application_deadline']-df['posting_date']).dt.days

Suspicious Score


In [33]:
df['suspicious_score'] = (
    df['keyword_flag'] * 2 +
    (df['has_website'] == 0).astype(int) +
    (df['desc_length'] < 200).astype(int)
)

In [39]:
df['employment_type']=df['employment_type'].str.lower()
df['education_level']=df['education_level'].str.lower()
df['location']=df['location'].str.lower()

df['telecommuting']=df['telecommuting'].astype(int)
df['has_logo']=df['has_logo'].astype(int)

In [35]:
df=df.drop_duplicates()

In [36]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   job_title                  3000 non-null   object        
 1   job_description            3000 non-null   object        
 2   requirements               3000 non-null   object        
 3   benefits                   3000 non-null   object        
 4   company_name               3000 non-null   object        
 5   company_profile            3000 non-null   object        
 6   industry                   3000 non-null   object        
 7   employment_type            3000 non-null   object        
 8   location                   3000 non-null   object        
 9   salary_range               3000 non-null   object        
 10  required_experience_years  3000 non-null   int64         
 11  education_level            3000 non-null   object        
 12  depart

,required_experience_years,posting_date,application_deadline,has_logo,num_open_positions,telecommuting,text_length,is_fake,desc_length,has_website,keyword_flag,suspicious_score
count,3000.000000,3000,3000,3000.00000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.0,3000.000000
mean,5.044000,2023-07-03 05:22:33.599999744,2024-07-01 17:33:36,0.24600,5.564000,0.507667,79.186667,0.490667,321.630667,0.509333,0.0,0.522000
min,0.000000,2023-01-01 00:00:00,2024-01-01 00:00:00,0.00000,1.000000,0.000000,69.000000,0.000000,181.000000,0.000000,0.0,0.000000
25%,2.000000,2023-04-03 00:00:00,2024-04-03 00:00:00,0.00000,3.000000,0.000000,69.000000,0.000000,219.000000,0.000000,0.0,0.000000
50%,5.000000,2023-07-03 00:00:00,2024-07-02 00:00:00,0.00000,6.000000,1.000000,89.000000,0.000000,384.000000,1.000000,0.0,0.000000
75%,8.000000,2023-10-02 00:00:00,2024-10-02 00:00:00,0.00000,8.000000,1.000000,89.000000,1.000000,421.000000,1.000000,0.0,1.000000
max,10.000000,2024-01-01 00:00:00,2024-12-31 00:00:00,1.00000,10.000000,1.000000,89.000000,1.000000,478.000000,1.000000,0.0,2.000000
std,3.168292,NaN,NaN,0.43075,2.890102,0.500025,9.999924,0.499996,102.309387,0.499996,0.0,0.558826


Checking class balance


In [41]:
df['is_fake'].value_counts(normalize=True)

is_fake
0    0.509333
1    0.490667
Name: proportion, dtype: float64

In [43]:
df.to_csv("C:\\Users\\Kirti\\OneDrive\\Desktop\\Fake Job Posting Detection & Analytics Dashboard\\data\\fake_real_job_postings.csv",index=False)